# Zor test ölçümü: Model A ve Model B

Gerçek kurumsal SMS'lerle iki eğitim senaryosunu karşılaştırır (bkz. `docs/zor-test.md`).
Veri `data/external/` altındadır ve git'e girmez. **Bu notebook'ta çıktı bırakmadan commit et.**

In [ ]:
import sys
sys.path.append("..")
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from src.temizle import Temizleyici

svm = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                            sublinear_tf=True, lowercase=False)),
    ("model", LinearSVC(class_weight="balanced", random_state=55)),
])

In [ ]:
# Eğitim verisi
train = pd.read_csv("../data/processed/tr_train.csv")
train = train[train["belirsiz"] == "h"].reset_index(drop=True)
X = train["mesaj"]
y = (train["ikili"] == "spam").astype(int)

# Zor test verisi
zor = pd.read_csv("../data/external/zor_gercek_hepsi.csv", keep_default_na=False)
zor = zor[zor["etiket"] != ""].reset_index(drop=True)      # belirsizleri at
y_zor = (zor["etiket"] != "meşru").astype(int)             # spam = reklam + dolandırıcılık

print(len(X), len(zor))
print(zor["etiket"].value_counts().to_dict())

In [ ]:
kisim = pd.Series(["1. kısım"] * 84 + ["2. kısım"] * (len(zor) - 84), index=zor.index)
print(zor.loc[kisim == "1. kısım", "etiket"].value_counts().to_dict())   # meşru 49, reklam 24, dolandırıcılık 11

Model A’da zor mesajlar eğitime hiç girmiyor, sadece sınav

In [ ]:
kisim = pd.Series(["1. kısım"] * 84 + ["2. kısım"] * (len(zor) - 84), index=zor.index)

def rapor(tahmin, ad):
    tahmin = pd.Series(tahmin, index=zor.index)
    for kisim_adi, maske in [("1. kısım", kisim == "1. kısım"),
                             ("2. kısım", kisim == "2. kısım"),
                             ("birleşik", kisim.notna())]:
        gercek = y_zor[maske]
        t = tahmin[maske]
        yanlis = ((t == 1) & (gercek == 0)).sum()
        yakalanan = ((t == 1) & (gercek == 1)).sum()
        print(f"{ad} | {kisim_adi:9s} | spam F1: {f1_score(gercek, t):.2f} | "
              f"yanlış alarm: {yanlis}/{(gercek == 0).sum()} | "
              f"yakalanan spam: {yakalanan}/{(gercek == 1).sum()}")

# Model A: yalnızca tr_train ile eğitilmiş
tahmin_A = clone(svm).fit(X, y).predict(zor["mesaj"])
rapor(tahmin_A, "Model A")

Model A’de gonderen sütununa göre dönüyoruz, her turda bir göndericiyi sınav yapıp kalanları eğitime ekliyoruz:

In [ ]:
tahmin_B = pd.Series(0, index=zor.index)

for g in zor["gonderen"].unique():
    sinav = zor["gonderen"] == g
    egit_X = pd.concat([X, zor.loc[~sinav, "mesaj"]], ignore_index=True)
    egit_y = pd.concat([y, y_zor[~sinav]], ignore_index=True)
    model = clone(svm).fit(egit_X, egit_y)
    tahmin_B[sinav] = model.predict(zor.loc[sinav, "mesaj"])

rapor(tahmin_B, "Model B")

## Beklenen sonuçlar (çıktılar temizlendiği için buraya yazıldı)

| Set | Model A (yalnız `tr_train`) | Model B (+ diğer göndericiler) |
|---|---|---|
| 1. kısım (84) | F1 0.65, yanlış alarm 28/49 | F1 0.85, yanlış alarm 8/49 |
| 2. kısım (49) | F1 0.90, yanlış alarm 7/19 | F1 0.92, yanlış alarm 3/19 |
| Birleşik (133) | F1 0.75, yanlış alarm 35/68 | F1 0.88, yanlış alarm 11/68 |

Küçük sapmalar (±0.02) SVM ayarları ya da veri sırası kaynaklı olabilir.